# Part 5 — The canonical SQL views (tutorial)

**This notebook is read-only.** It walks through each view's design rationale and shows a sample of its output. To actually build the views, run:

```bash
uv run iran-shock-ingest-prices   # fetches Brent + GB MID + TTF to data/raw/prices/
uv run iran-shock-ingest-gdelt    # downloads GDELT files (long-running)
uv run iran-shock-build            # loads raw -> DuckDB, executes sql/v_*.sql in order
```

The canonical view definitions live in `sql/v_*.sql` — that's the source of truth. The build script reads those files; this notebook reads the built views in **read-only mode**.

Seven views, in composition order:

| View | Question it answers | Depends on |
|---|---|---|
| `v_middle_east_events` | per-day conflict signal (count, tone, severity, mentions), windowed to project dates | `gdelt` |
| `v_event_geo` | per-place rollup for the map, windowed to project dates | `gdelt` |
| `v_brent_daily` | passthrough of Brent daily closes | `brent` |
| `v_gb_power_daily` | passthrough of GB MID (APX) daily mean | `gb_power` |
| `v_ttf_daily` | passthrough of TTF (European gas) daily closes | `ttf` |
| `v_combined_daily` | four series on one trading-day calendar (Brent + TTF + GB power + news) | the above |
| `v_event_impact` | each curated event vs Brent/TTF/power at +3 trading days (nearest-trading-day) | `timeline` + price views |

Three design ideas span all seven:

- **Casts at read time, not load time.** Build loaded everything as VARCHAR. Views cast where they need to. A single malformed value can't poison whole-column types.
- **Anchor the time series on Brent's trading days.** Brent has no weekends, no holidays. Anchoring `v_combined_daily` LEFT JOINs on Brent makes the % change chart continuous.
- **Window the GDELT views.** GDELT files contain backfilled events from earlier dates. `v_middle_east_events` and `v_event_geo` filter on `BETWEEN '2026-02-01' AND '2026-05-31'` so only project-window events count.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

In [2]:
import duckdb

from iran_shock.config import DUCKDB_PATH

con = duckdb.connect(DUCKDB_PATH, read_only=True)

## `v_middle_east_events` — daily aggregate of conflict events

**Question:** for each day, how much (and how negative) was the Middle East conflict news?

**Design choices:**
- **One row per day** — the join key with prices.
- **Four metrics** — different lenses on the same day: `event_count` (volume), `avg_tone` (sentiment), `avg_goldstein` (type-weighted destabilisation), `total_mentions` (coverage loudness).
- **Filter mirrors Part 3** — `QuadClass IN (3,4)` ∩ FIPS Middle East. Kept in step with `config.MIDDLE_EAST`.
- **Windowed** to the project dates so backfilled events from earlier years don't leak in.
- **DuckDB `strptime` not Athena's `date_parse`** — the only swap when this ports to Athena in Stage 3.

In [3]:
print(Path("sql/v_middle_east_events.sql").read_text())

-- Athena port: strptime(...) -> date_parse(...);  ::TYPE -> CAST(... AS TYPE)
CREATE OR REPLACE VIEW v_middle_east_events AS
SELECT strptime(day,'%Y%m%d')::DATE AS event_date,
       count(*) AS event_count,
       avg(avgtone::DOUBLE) AS avg_tone,
       avg(goldsteinscale::DOUBLE) AS avg_goldstein,
       sum(nummentions::INT) AS total_mentions
FROM gdelt
WHERE actiongeo_countrycode IN ('IR','IS','US','SA','AE','QA','IZ','KU','BA','LE','SY','YM')
  AND quadclass IN ('3','4')
  AND strptime(day,'%Y%m%d')::DATE BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
GROUP BY 1 ORDER BY 1;
-- NOTE: the IN-list mirrors config.MIDDLE_EAST — keep the two in step.



In [4]:
df = con.execute("SELECT * FROM v_middle_east_events ORDER BY event_date").df()
print(df.shape)
df.head()

(150, 5)


,event_date,event_count,avg_tone,avg_goldstein,total_mentions
0,2026-01-01,7274,-4.572285,-5.922725,32010.0
1,2026-01-02,9827,-4.959848,-5.987758,42630.0
2,2026-01-03,9856,-4.951744,-5.668608,41068.0
3,2026-01-04,8845,-4.710265,-5.848344,34964.0
4,2026-01-05,13548,-4.825264,-5.767988,54467.0


## `v_event_geo` — per-place rollup for the map

**Question:** where (lat/long) is the conflict news happening, and how negative / loud?

**Design choices:**
- **`ActionGeo_Type` 3/4 only** — city/landmark resolution. Types 1/2/5 are centroids; events pile up in country middles.
- **Lat/long rounded to 1 decimal** — nearby events aggregate into fewer, heavier dots without a spatial library.
- **Drop empty coordinates** — some rows have `''` for lat/long; can't plot.
- **Same window** as `v_middle_east_events`.

In [5]:
print(Path("sql/v_event_geo.sql").read_text())

CREATE OR REPLACE VIEW v_event_geo AS
SELECT actiongeo_fullname AS place,
       actiongeo_countrycode AS fips_country,
       round(actiongeo_lat::DOUBLE, 1) AS lat,
       round(actiongeo_long::DOUBLE, 1) AS lon,
       count(*) AS event_count,
       avg(avgtone::DOUBLE) AS avg_tone,
       sum(nummentions::INT) AS total_mentions
FROM gdelt
WHERE quadclass IN ('3','4')
  AND actiongeo_type IN ('3','4')
  AND actiongeo_lat <> '' AND actiongeo_long <> ''
  AND strptime(day,'%Y%m%d')::DATE BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
GROUP BY 1,2,3,4;



In [6]:
df = con.execute("SELECT * FROM v_event_geo").df()
print(df.shape)
df.sort_values("event_count", ascending=False).head(10)

(75113, 7)


,place,fips_country,lat,lon,event_count,avg_tone,total_mentions
32903,"Tehran, Tehran, Iran",IR,35.8,51.5,130989,-4.887427,540225.0
28810,"Washington, District of Columbia, United States",US,38.9,-77.0,90573,-3.945100,357314.0
28806,"Gaza, Israel (general), Israel",IS,31.4,34.3,84478,-5.179395,332282.0
37000,"White House, District of Columbia, United States",US,38.9,-77.0,53735,-3.531662,205708.0
16384,"London, London, City of, United Kingdom",UK,51.5,-0.1,43197,-3.557225,213703.0
12289,"Moscow, Moskva, Russia",RS,55.8,37.6,33371,-4.662239,129162.0
30859,"Jerusalem, Israel (general), Israel",IS,31.8,35.2,32834,-5.408874,131241.0
36997,"Beijing, Beijing, China",CH,39.9,116.4,28185,-2.762695,120263.0
59270,"Delhi, Delhi, India",IN,28.7,77.2,22435,-5.219709,102028.0
16385,"Kyiv, Kyyiv, Misto, Ukraine",UP,50.4,30.5,22279,-4.442570,83289.0


## `v_brent_daily`, `v_gb_power_daily`, `v_ttf_daily` — price passthroughs

**Question:** the raw price tables already have one row per day. Why a view?

**Separation of concerns.** Every consumer reads from `v_*` views, never raw tables. The day `yfinance` schemas drift or Elexon renames a field, **only the view changes** — every downstream query keeps working. A small abstraction now buys back an hour later. These views are also where derived columns belong (e.g. a `pct_change` column on Brent).

In [7]:
for v in ("v_brent_daily", "v_gb_power_daily", "v_ttf_daily"):
    df = con.execute(f"SELECT * FROM {v} ORDER BY price_date").df()
    print(f"{v}: {df.shape[0]} rows  ({df['price_date'].min()} -> {df['price_date'].max()})")
    print(df.head(3))
    print()

v_brent_daily: 102 rows  (2026-01-02 00:00:00 -> 2026-05-29 00:00:00)
  price_date  brent_close
0 2026-01-02    60.750000
1 2026-01-05    61.759998
2 2026-01-06    60.700001

v_gb_power_daily: 150 rows  (2026-01-01 00:00:00 -> 2026-05-30 00:00:00)
  price_date  gb_power_avg
0 2026-01-01     54.954167
1 2026-01-02     80.396667
2 2026-01-03     81.766250

v_ttf_daily: 102 rows  (2026-01-02 00:00:00 -> 2026-05-29 00:00:00)
  price_date  ttf_close
0 2026-01-02  29.004000
1 2026-01-05  27.399000
2 2026-01-06  28.072001



## `v_combined_daily` — four series on one timeline

**Question:** one row per day with Brent, TTF gas, GB power, and the news signal — the spine of the hero chart.

**Design choices:**
- **Anchor on Brent's trading days.** No weekends, no holidays. Anchoring elsewhere puts gaps in the % change chart where markets were closed.
- **LEFT JOINs.** Brent trades on days when nothing newsworthy happens; we still want a row, with NULL for `event_count`. The chart degrades gracefully — a flat line, not a missing one.
- **No casts here.** We pull from the typed views above, not raw `gdelt` / `brent` — join keys are already DATEs and metrics already DOUBLE/INT. Each layer narrows the contract.

In [8]:
print(Path("sql/v_combined_daily.sql").read_text())

CREATE OR REPLACE VIEW v_combined_daily AS
SELECT b.price_date AS d,
       b.brent_close,
       g.ttf_close,
       p.gb_power_avg,
       e.event_count,
       e.avg_tone
FROM v_brent_daily b
LEFT JOIN v_ttf_daily         g ON b.price_date = g.price_date
LEFT JOIN v_gb_power_daily    p ON b.price_date = p.price_date
LEFT JOIN v_middle_east_events e ON b.price_date = e.event_date
ORDER BY d;



In [9]:
df = con.execute("SELECT * FROM v_combined_daily ORDER BY d").df()
print(df.shape)
df.head(10)

(102, 6)


,d,brent_close,ttf_close,gb_power_avg,event_count,avg_tone
0,2026-01-02,60.750000,29.004000,80.396667,9827,-4.959848
1,2026-01-05,61.759998,27.399000,120.841667,13548,-4.825264
2,2026-01-06,60.700001,28.072001,147.029583,15185,-4.860023
3,2026-01-07,59.959999,28.777000,78.836250,15850,-4.696782
4,2026-01-08,61.990002,27.698999,144.991250,16993,-4.997963
5,2026-01-09,63.340000,28.374001,88.093750,17307,-4.955392
6,2026-01-12,63.869999,30.246000,85.469792,14109,-4.972168
7,2026-01-13,65.470001,31.473000,91.758750,17365,-5.034223
8,2026-01-14,66.519997,31.813000,90.418542,17858,-5.009615
9,2026-01-15,63.759998,33.158001,96.996042,18287,-4.903945


## `v_event_impact` — the event study

**Question:** for each curated timeline event, what did Brent, TTF gas, and GB power do 3 trading days later?

**Design choices:**
- **+3 days is the standard short-horizon window** in commodity event studies — long enough for narrative shocks to be priced in, short enough that other drivers haven't taken over.
- **Nearest-trading-day lookup** via correlated subqueries. Weekend events (28 Feb Sat, 1 Mar Sun) still produce clean numbers without distorting the real `event_date` in the CSV.
- **Three side-by-side `pct_3d` columns** (Brent, TTF, power) make the transmission test legible: where all three move together, the chain is visible; where TTF moves but Brent doesn't (e.g. Qatar LNG strikes), gas-as-mechanism stands out.
- **`+ INTERVAL 3 DAY` is the DuckDB form**; Athena port is `date_add('day', 3, t.event_date)`.

In [10]:
print(Path("sql/v_event_impact.sql").read_text())

-- Athena port: "+ INTERVAL 3 DAY" -> date_add('day',3,t.event_date)
--
-- Date anchors come from Brent (futures, no weekends). Power has weekend data
-- in Elexon, but if we let each series pick its own nearest date we get
-- mismatched windows -- a Sun event would look up Sun's low-demand power vs
-- Mon's Brent, generating spurious +X% moves that are actually weekday-vs-
-- weekend seasonality. Anchoring all three on Brent's calendar fixes this.
CREATE OR REPLACE VIEW v_event_impact AS
WITH anchored AS (
    SELECT t.event_date, t.label,
           (SELECT MIN(price_date) FROM v_brent_daily
              WHERE price_date >= t.event_date) AS t0_date,
           (SELECT MIN(price_date) FROM v_brent_daily
              WHERE price_date >= t.event_date + INTERVAL 3 DAY) AS t3_date
    FROM timeline t
),
e AS (
    SELECT a.event_date, a.label,
           (SELECT brent_close FROM v_brent_daily WHERE price_date = a.t0_date) AS brent_t,
           (SELECT brent_close FROM v_brent_daily WHE

In [11]:
df = con.execute("SELECT * FROM v_event_impact ORDER BY event_date").df()
print(df.shape)
df

(15, 11)


,event_date,label,brent_t,brent_t3,brent_pct_3d,ttf_t,ttf_t3,ttf_pct_3d,power_t,power_t3,power_pct_3d
0,2026-02-28,Operation Epic Fury begins,77.739998,81.400002,4.7,44.506001,54.290001,22.0,72.247292,116.579583,61.4
1,2026-03-01,Iran retaliates; first US dead,77.739998,81.400002,4.7,44.506001,48.766998,9.6,72.247292,120.833333,67.2
2,2026-03-05,Hormuz paralysis starts,85.410004,98.959999,15.9,50.730999,56.452999,11.3,107.279375,135.282500,26.1
3,2026-03-07,Israeli strikes hit Tehran/Alborz fuel depots,98.959999,87.800003,-11.3,56.452999,47.393002,-16.0,135.282500,91.252292,-32.5
4,2026-03-12,KC-135 loss and tanker attacks,100.459999,100.209999,-0.2,50.869999,50.887001,0.0,87.615532,102.176250,16.6
5,2026-03-18,Iran strikes Qatar LNG infrastructure,107.379997,99.940002,-6.9,54.661999,56.682999,3.7,110.009167,122.262500,11.1
6,2026-03-24,US 15-point settlement proposal,104.489998,112.570000,7.7,54.041000,54.176998,0.3,58.016596,98.973958,70.6
7,2026-03-31,Iran strikes loaded tanker off Dubai,118.349998,109.769997,-7.2,50.757000,50.043999,-1.4,115.944792,98.050244,-15.4
8,2026-04-02,Trump vows harder phase of attacks,109.029999,109.769997,0.7,50.043999,50.043999,0.0,102.084792,98.050244,-4.0
9,2026-04-07,Pakistan-brokered ceasefire announced,109.269997,95.199997,-12.9,53.247002,43.638000,-18.0,80.156053,83.109167,3.7


## Sanity sweep

Same check the build script runs at the end — every view should return rows. If any return 0, re-check the ingest CLIs and re-run the build.

In [12]:
for v in [
    "v_middle_east_events",
    "v_event_geo",
    "v_brent_daily",
    "v_gb_power_daily",
    "v_ttf_daily",
    "v_combined_daily",
    "v_event_impact",
]:
    try:
        n = con.execute(f"SELECT count(*) FROM {v}").fetchone()[0]
        print(f"{v:<25} {n:>10,} rows")
    except Exception as e:
        print(f"{v:<25} ERROR — {e}")

v_middle_east_events             150 rows
v_event_geo                   75,113 rows
v_brent_daily                    102 rows
v_gb_power_daily                 150 rows
v_ttf_daily                      102 rows
v_combined_daily                 102 rows
v_event_impact                    15 rows
